# Portfolio Backtesting System - Main Application

Interactive Jupyter Notebook for portfolio backtesting and analysis.

**Features:**
- SQL-focused analytics
- CRUD operations for scenarios
- Benchmark comparison
- Data visualization
- Export to text files

## Quick Start Guide

1. **Setup:** Run the setup cell below
2. **Explore:** Browse ETFs and benchmarks
3. **Analyze:** Run 11 analytics insights
4. **Create:** Build your own portfolios
5. **Compare:** Benchmark your performance

## Setup and Configuration

In [ ]:
# Import all necessary modules
from database import db
from analytics import analytics
from config import *
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import os

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Create directories
for directory in [BACKUP_DIR, EXPORT_DIR]:
    os.makedirs(directory, exist_ok=True)

# Connect to database
if db.connect():
    print("✅ Portfolio Backtesting System Ready!")
    print(f"   Database: {MYSQL_CONFIG['database']}")
    print(f"   Version: {APP_VERSION}")
else:
    print("❌ Database connection failed!")

## 📊 Section 1: Data Exploration

### 1.1 View All ETFs

In [ ]:
# Get all ETFs
etfs = db.get_all_etfs()
etf_df = pd.DataFrame(etfs)

print(f"📊 Total ETFs: {len(etf_df)}\n")
display(etf_df.head(15))

# Distribution by asset class
print("\n📈 Distribution by Asset Class:")
asset_counts = etf_df['asset_class'].value_counts()
print(asset_counts)

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.pie(asset_counts.values, labels=asset_counts.index, autopct='%1.0f%%')
ax1.set_title('ETF Distribution by Asset Class')

ax2.bar(asset_counts.index, asset_counts.values)
ax2.set_xlabel('Asset Class')
ax2.set_ylabel('Count')
ax2.set_title('ETF Count by Asset Class')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 1.2 View All Benchmark Portfolios

In [ ]:
# Get all benchmarks
benchmarks = db.get_all_benchmarks()
bench_df = pd.DataFrame(benchmarks)

print(f"📊 Total Benchmarks: {len(bench_df)}\n")
display(bench_df[['benchmark_id', 'benchmark_name', 'risk_level', 'target_return']].head(20))

# Distribution by risk level
print("\n📈 Distribution by Risk Level:")
risk_counts = bench_df['risk_level'].value_counts()
print(risk_counts)

# Visualization
plt.figure(figsize=(8, 5))
plt.pie(risk_counts.values, labels=risk_counts.index, autopct='%1.0f%%')
plt.title('Benchmark Distribution by Risk Level')
plt.show()

### 1.3 View Benchmark Holdings

In [ ]:
# Select a benchmark to view holdings
benchmark_id = 1  # Change this

holdings = db.get_benchmark_holdings(benchmark_id)
if holdings:
    holdings_df = pd.DataFrame(holdings)
    print(f"📊 Holdings for: {holdings_df['benchmark_name'].iloc[0]}\n")
    display(holdings_df[['ticker_symbol', 'etf_name', 'target_weight']])
    
    # Pie chart
    plt.figure(figsize=(10, 6))
    plt.pie(holdings_df['target_weight'], labels=holdings_df['ticker_symbol'], autopct='%1.1f%%')
    plt.title(f"Asset Allocation: {holdings_df['benchmark_name'].iloc[0]}")
    plt.show()
else:
    print("❌ No holdings found")

## 📈 Section 2: Analytics Insights

### 2.1 Best Performing ETFs

In [ ]:
results = analytics.get_best_performing_etfs(limit=10)
df = pd.DataFrame(results)

print("📊 TOP 10 BEST PERFORMING ETFs\n")
display(df[['ticker_symbol', 'etf_name', 'asset_class', 'total_return_pct']])

# Bar chart
plt.figure(figsize=(12, 6))
plt.barh(df['ticker_symbol'], df['total_return_pct'], color='green')
plt.xlabel('Total Return (%)')
plt.title('Top 10 Best Performing ETFs')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### 2.2 Volatility & Risk Analysis

In [ ]:
results = analytics.get_etf_volatility(limit=20)
df = pd.DataFrame(results)

print("📊 VOLATILITY ANALYSIS\n")
display(df[['ticker_symbol', 'etf_name', 'annualized_volatility_pct', 'annualized_return_pct']].head(10))

# Risk-Return scatter
plt.figure(figsize=(12, 7))
plt.scatter(df['annualized_volatility_pct'], df['annualized_return_pct'], s=100, alpha=0.6)
for idx, row in df.iterrows():
    plt.annotate(row['ticker_symbol'], 
                (row['annualized_volatility_pct'], row['annualized_return_pct']),
                fontsize=8, alpha=0.7)
plt.xlabel('Annualized Volatility (%)')
plt.ylabel('Annualized Return (%)')
plt.title('Risk vs Return Profile')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 2.3 Asset Class Performance

In [ ]:
results = analytics.get_asset_class_performance()
df = pd.DataFrame(results)

print("📊 ASSET CLASS PERFORMANCE\n")
display(df)

# Bar chart
fig, ax = plt.subplots(figsize=(12, 6))
x = range(len(df))
width = 0.35

ax.barh([i-width/2 for i in x], df['avg_return_pct'], width, label='Avg Return')
ax.barh([i+width/2 for i in x], df['max_return_pct'], width, label='Max Return', alpha=0.7)

ax.set_yticks(x)
ax.set_yticklabels(df['asset_class'])
ax.set_xlabel('Return (%)')
ax.set_title('Performance by Asset Class')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.4 Concentration Risk Analysis

In [ ]:
results = analytics.analyze_concentration_risk()
df = pd.DataFrame(results)

print("📊 CONCENTRATION RISK (HHI Index)\n")
display(df[['benchmark_name', 'risk_level', 'num_holdings', 'hhi_index', 'concentration_level']].head(15))

# Visualization
top_10 = df.head(10)
plt.figure(figsize=(12, 6))
colors = ['red' if x > 0.25 else 'orange' if x > 0.15 else 'green' for x in top_10['hhi_index']]
plt.barh(top_10['benchmark_name'], top_10['hhi_index'], color=colors)
plt.xlabel('HHI Index')
plt.title('Top 10 Most Concentrated Portfolios')
plt.axvline(x=0.25, color='red', linestyle='--', alpha=0.5, label='High Risk')
plt.axvline(x=0.15, color='orange', linestyle='--', alpha=0.5, label='Medium Risk')
plt.legend()
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 🎯 Section 3: Scenario Management (CRUD)

### 3.1 Create New Scenario

In [ ]:
# Define your scenario
scenario_data = {
    'benchmark_id': 1,  # Traditional 60/40
    'created_by': 'Your Name',
    'scenario_name': 'My Test Portfolio',
    'initial_capital': 100000.00,
    'start_date': '2020-01-01',
    'end_date': '2024-12-31',
    'strategy_type': 'BUY_HOLD',
    'rebalance_freq': None,
    'monthly_contribution': 0
}

# Create scenario
scenario_id = db.create_scenario(scenario_data)

if scenario_id:
    print(f"✅ Scenario created! ID: {scenario_id}")
    
    # Add holdings (example: 60/40 portfolio)
    holdings = [
        ('SPY', 0.60),
        ('AGG', 0.40)
    ]
    
    for ticker, weight in holdings:
        etf = db.get_etf_by_ticker(ticker)
        if etf:
            db.add_scenario_holding(scenario_id, etf['etf_id'], weight)
            print(f"   Added {ticker}: {weight:.1%}")
    
    db.connection.commit()
    print(f"\n✅ Scenario {scenario_id} ready!")
else:
    print("❌ Failed to create scenario")

### 3.2 View All Scenarios

In [ ]:
scenarios = db.get_all_scenarios()

if scenarios:
    df = pd.DataFrame(scenarios)
    print(f"📊 Total Scenarios: {len(df)}\n")
    display(df[['scenario_id', 'scenario_name', 'created_by', 'strategy_type', 'initial_capital', 'start_date', 'end_date']])
else:
    print("❌ No scenarios found")

### 3.3 View Scenario Details

In [ ]:
scenario_id = 1  # Change this

scenario = db.get_scenario(scenario_id)
if scenario:
    print(f"📊 Scenario {scenario_id}: {scenario['scenario_name']}\n")
    
    for key, value in scenario.items():
        print(f"   {key:20s}: {value}")
    
    print("\n📈 Holdings:")
    holdings = db.get_scenario_holdings(scenario_id)
    if holdings:
        df = pd.DataFrame(holdings)
        display(df[['ticker_symbol', 'etf_name', 'target_weight']])
        
        # Pie chart
        plt.figure(figsize=(8, 8))
        plt.pie(df['target_weight'], labels=df['ticker_symbol'], autopct='%1.1f%%')
        plt.title(f"Asset Allocation: {scenario['scenario_name']}")
        plt.show()
else:
    print("❌ Scenario not found")

## 📤 Section 4: Export Results

In [ ]:
# Export best performing ETFs to text file
results = analytics.get_best_performing_etfs(limit=20)
if results:
    df = pd.DataFrame(results)
    
    filename = f"{EXPORT_DIR}/best_etfs_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    
    with open(filename, 'w') as f:
        f.write(f"{APP_NAME}\n")
        f.write(f"Export Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("=" * 80 + "\n\n")
        f.write("BEST PERFORMING ETFs\n\n")
        f.write(df.to_string())
    
    print(f"✅ Exported to: {filename}")
else:
    print("❌ No data to export")

## 💾 Section 5: Backup Database Info

In [ ]:
# Backup database statistics
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f"{BACKUP_DIR}/db_backup_{timestamp}.txt"

tables = [
    'etf_master', 'price_history', 'benchmark_portfolios',
    'benchmark_holdings', 'backtest_scenarios', 'scenario_holdings',
    'backtest_results', 'portfolio_snapshots', 'transaction_log'
]

with open(filename, 'w') as f:
    f.write(f"{APP_NAME} - Database Backup\n")
    f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("=" * 80 + "\n\n")
    
    f.write("TABLE STATISTICS\n")
    f.write("-" * 80 + "\n")
    
    for table in tables:
        result = db.execute_query(f"SELECT COUNT(*) as count FROM {table}")
        if result:
            count = result[0]['count']
            f.write(f"{table:30s} {count:>10,} rows\n")

print(f"✅ Backup created: {filename}")

## 🎉 System Ready!

You can now:
- ✅ Explore ETFs and benchmarks
- ✅ Run analytics insights
- ✅ Create and manage scenarios
- ✅ Compare vs benchmarks
- ✅ Export results

**Next Steps:**
1. Create your own scenarios
2. Run backtests
3. Compare with benchmarks
4. Prepare presentation

## Cleanup (Optional)

In [ ]:
# Uncomment to close database connection
# db.disconnect()